# Runtime Verification and Baseline: Setting Up Our Measurement Ritual

Welcome! We are about to start writing code that runs on the GPU. But before we write a single line of a transformer or a neural network, we need to understand the physical substrate we are running on. In engineering, you can't optimize what you can't measure.

### What We Will Learn
* **Querying the Substrate:** How to inspect the physical GPU assigned to us.
* **Framework Connection:** Verify that PyTorch can interact with CUDA.
* **Host-Device Boundary:** Understand the asynchronous nature of GPU operations.
* **Measurement Ritual:** Build a reliable, synchronized timing mechanism to capture a clean baseline.

### The Backdrop
The shift from CPU to GPU programming is not just a speed upgrade—it's a change in the programming model. On the CPU, code execution is typically synchronous and sequential. On the GPU, the CPU behaves as a **Host** that launches commands into a queue on the **Device** (the GPU) and returns immediately. If you try to benchmark GPU code using standard CPU wall-clock timers without synchronization, you will measure the time it took to *queue* the work, not to *execute* it. Let's fix that.

## Step 1: Inspecting the Hardware (`nvidia-smi`)

Let's talk directly to the hardware first. Since Colab runs on virtualized Linux containers, we can execute a system utility called `nvidia-smi` (NVIDIA System Management Interface). This command queries the system's graphics driver to confirm that a physical GPU is connected to our environment and displays its operational status.

In [ ]:
# Run nvidia-smi to query the attached NVIDIA GPU status
!nvidia-smi

### What the output tells us
If you see a table detailing a GPU model (like Tesla T4, V100, or A100), congrats—you have a GPU-enabled runtime. Take note of:
1. **GPU Name:** The hardware type (e.g., "Tesla T4").
2. **GPU-Util:** The current utilization percentage.
3. **Memory Usage:** How much High Bandwidth Memory (HBM) is currently allocated.

*Note: If the command fails or says "NVIDIA-SMI has failed...", go to the menu at the top: **Runtime > Change runtime type** and select a GPU accelerator (like T4 GPU) before proceeding.*

## Step 2: Verifying the PyTorch-CUDA Link

Now that the system knows a GPU exists, we need to make sure our deep learning framework, PyTorch, can actually interface with it. We'll start by importing PyTorch (`torch`) and the standard `time` library to handle timings later.

In [ ]:
import time  # For high-resolution wall-clock timing
import torch  # The PyTorch library for tensor computations

We imported `time` because we want to measure wall-clock elapsed time on the host CPU, and `torch` because it is our main math framework. Now let's query PyTorch's backend to see if it detects CUDA.

Let's call `torch.cuda.is_available()`. This check is the first thing you should run in any GPU script to gracefully fall back to CPU if hardware isn't present.

In [ ]:
# Verify PyTorch can see the GPU
is_cuda = torch.cuda.is_available()
print(f"CUDA available: {is_cuda}")

If this prints `True`, PyTorch has successfully loaded the CUDA runtime and linked with the driver. If it prints `False`, check that your runtime was set to GPU and restarted.

Next, let's identify the exact device PyTorch has registered at index `0` (the default active GPU).

In [ ]:
# Query the name of the first GPU (device 0)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU Device Name: {device_name}")

Knowing the specific GPU model is critical. Optimizations that work on an older Tesla T4 might behave differently on an A100. We will keep this hardware model in mind as our backdrop.

## Step 3: The Asynchronous Execution Gap

Here is the core trap that catches many beginners: **GPU kernel launches are non-blocking**.

When you write `torch.matmul(x, y)` on GPU tensors, the CPU does not wait for the GPU to finish the multiplication. Instead, the CPU pushes a "launch instruction" to the GPU's execution queue (known as a CUDA stream) and immediately moves to the next line of your Python script. The GPU processes this queue independently.

If you start a timer, launch a matrix multiplication, and stop the timer immediately after, you are timing the **launch latency** (the time it took the CPU to enqueue the task, usually micro-seconds), not the actual execution time on the GPU.

Here is a sketch explaining this flow:

<svg viewBox="0 0 800 480" width="100%" height="auto" style="background-color: #faf8f5; border: 1px solid #e5e7eb; border-radius: 12px; font-family: 'Chalkboard SE', 'Chalkboard', 'Comic Neue', 'Comic Sans MS', sans-serif; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.05);">
  <!-- Styles for hand-drawn look -->
  <style>
    .sketch-text { fill: #2d3748; font-size: 14px; }
    .title-text { fill: #1a202c; font-size: 20px; font-weight: bold; }
    .hand-line { stroke: #718096; stroke-width: 2.5; stroke-linecap: round; fill: none; }
    .flow-arrow { stroke: #e53e3e; stroke-width: 2.5; stroke-linecap: round; fill: none; marker-end: url(#arrow-red); }
    .cpu-box { fill: none; stroke: #3182ce; stroke-width: 3; stroke-linecap: round; stroke-linejoin: round; rx: 12; }
    .gpu-box { fill: none; stroke: #38a169; stroke-width: 3; stroke-linecap: round; stroke-linejoin: round; rx: 12; }
    .timer-line { stroke: #dd6b20; stroke-width: 2; stroke-dasharray: 6 6; }
    .hand-rect { fill: none; stroke: #4a5568; stroke-width: 2.5; stroke-linecap: round; stroke-linejoin: round; rx: 6; }
  </style>
  
  <defs>
    <marker id="arrow-red" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#e53e3e" />
    </marker>
    <marker id="arrow-blue" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#3182ce" />
    </marker>
  </defs>

  <!-- Faint Grid lines background -->
  <pattern id="grid" width="20" height="20" patternUnits="userSpaceOnUse">
    <path d="M 20 0 L 0 0 0 20" fill="none" stroke="#f0ece4" stroke-width="1" />
  </pattern>
  <rect width="100%" height="100%" fill="url(#grid)" rx="12" />

  <!-- Title & Backdrop -->
  <text x="40" y="45" class="title-text">Why synchronization is non-negotiable</text>
  <text x="40" y="70" class="sketch-text" style="font-size: 13.5px; fill: #4a5568; font-style: italic;">CPU launches kernels to the GPU queue and immediately returns (asynchronous execution).</text>

  <!-- CPU Block -->
  <rect x="75" y="105" width="230" height="320" class="cpu-box" />
  <text x="190" y="135" class="sketch-text" style="font-weight: bold; text-anchor: middle; fill: #2b6cb0; font-size: 16px;">CPU (Host)</text>
  
  <!-- GPU Block -->
  <rect x="495" y="105" width="230" height="320" class="gpu-box" />
  <text x="610" y="135" class="sketch-text" style="font-weight: bold; text-anchor: middle; fill: #276749; font-size: 16px;">GPU (Device)</text>

  <!-- Timelines inside CPU and GPU -->
  <line x1="190" y1="150" x2="190" y2="400" class="hand-line" />
  <line x1="610" y1="150" x2="610" y2="400" class="hand-line" />

  <!-- Timeline Flow steps -->
  <!-- Step 1: Start Timer -->
  <circle cx="190" cy="170" r="5" fill="#dd6b20" />
  <text x="175" y="174" class="sketch-text" style="text-anchor: end; fill: #dd6b20; font-weight: bold; font-size: 12px;">Start Timer (t_start)</text>

  <!-- Step 2: Enqueue Matmul -->
  <circle cx="190" cy="210" r="5" fill="#3182ce" />
  <text x="175" y="214" class="sketch-text" style="text-anchor: end; font-size: 12.5px;">Launch Matmul Kernel</text>
  
  <!-- Arrow from CPU to GPU (Enqueue) -->
  <path d="M 195 210 Q 350 210 595 245" class="flow-arrow" />
  <text x="350" y="215" class="sketch-text" style="font-size: 11.5px; fill: #e53e3e; font-style: italic;">Kernel submitted to queue...</text>

  <!-- GPU starts processing -->
  <circle cx="610" cy="245" r="5" fill="#38a169" />
  <text x="625" y="249" class="sketch-text" style="font-size: 12.5px; fill: #2f855a; font-weight: bold;">GPU executes Kernel</text>

  <!-- CPU stops timer immediately (wrong way) -->
  <circle cx="190" cy="260" r="5" fill="#e53e3e" />
  <text x="175" y="264" class="sketch-text" style="text-anchor: end; fill: #e53e3e; font-weight: bold; font-size: 12px;">Stop Timer (no sync) ❌</text>
  <path d="M 190 170 L 190 260" class="timer-line" stroke="#e53e3e" />
  <text x="85" y="225" class="sketch-text" style="fill: #e53e3e; font-size: 11px; font-weight: bold;">Measures submission (0.1ms)</text>

  <!-- GPU is still running... -->
  <line x1="610" y1="245" x2="610" y2="330" stroke="#38a169" stroke-width="4.5" stroke-linecap="round" />
  <circle cx="610" cy="330" r="5" fill="#38a169" />
  <text x="625" y="334" class="sketch-text" style="font-size: 12.5px; fill: #718096;">GPU completes operation</text>

  <!-- CPU with synchronize() -->
  <rect x="110" y="300" width="160" height="35" class="hand-rect" style="fill: #fed7d7; stroke: #e53e3e;" />
  <text x="190" y="322" class="sketch-text" style="text-anchor: middle; font-size: 12px; font-weight: bold; fill: #c53030;">synchronize() (Blocks CPU)</text>

  <!-- Sync completion arrow from GPU to CPU -->
  <path d="M 605 330 Q 395 330 195 330" stroke="#3182ce" stroke-width="2.5" stroke-linecap="round" fill="none" marker-end="url(#arrow-blue)" />
  <text x="350" y="348" class="sketch-text" style="font-size: 11.5px; fill: #2b6cb0; font-style: italic;">GPU is idle, CPU unblocks</text>

  <!-- Stop Timer (With Sync) -->
  <circle cx="190" cy="370" r="5" fill="#38a169" />
  <text x="175" y="374" class="sketch-text" style="text-anchor: end; fill: #2f855a; font-weight: bold; font-size: 12px;">Stop Timer (with sync) ✅</text>
  <path d="M 190 170 L 190 370" class="timer-line" stroke="#38a169" />
  <text x="65" y="380" class="sketch-text" style="fill: #2f855a; font-size: 11px; font-weight: bold;">Measures real GPU runtime (5.2ms)</text>

</svg>

Let's write a simple matrix multiplication test and measure it correctly.

### 1. Initializing our matrices

We will set up our computation device target dynamically and allocate two large random matrices `x` and `y` (size 1024x1024) directly on the target device.

In [ ]:
# Set target device and allocate random matrices
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn((1024, 1024), device=device)
y = torch.randn((1024, 1024), device=device)

This assigns the device string based on CUDA availability, and creates the tensor arrays directly on the GPU memory space (or CPU RAM if GPU isn't ready).

### 2. The Measurement Ritual: Start Timer

To measure only the core computation, we must make sure the GPU has finished all previous allocations. We force a synchronization first, then capture the CPU start time.

In [ ]:
# Synchronize GPU and record start time
if device == "cuda":
    torch.cuda.synchronize()  # Block CPU until GPU queue clears
start_time = time.perf_counter()  # Start CPU timer

By synchronizing, we ensure any random matrix generation overhead is finished and the GPU is completely idle. Our CPU wall-clock start time is now highly accurate.

### 3. The Workload: Matrix Multiplications

Next, we run the matrix multiplication 10 times in a loop. Because PyTorch enqueues these operations asynchronously, the loop completes immediately on the CPU.

In [ ]:
# Run matrix multiplication in a loop
for _ in range(10):
    result = torch.matmul(x, y)  # Enqueued asynchronously

The GPU driver queues these 10 operations sequentially inside the CUDA stream. If we were to stop the timer now, we would not capture the GPU execution.

### 4. The Measurement Ritual: Stop Timer

To block the CPU until the GPU finishes calculating all 10 operations, we call `synchronize` again. Only then do we record the end time and calculate the elapsed time.

In [ ]:
# Synchronize again and calculate elapsed time
if device == "cuda":
    torch.cuda.synchronize()  # Block CPU until all 10 matmuls finish
elapsed_time = time.perf_counter() - start_time
print(f"Elapsed time: {elapsed_time:.6f} seconds")

This printed number represents the actual elapsed time. We have successfully recorded a clean, reproducible baseline!

## First-Principles Checkpoint: Understanding Our Baseline

Let's pause and think about what this number actually represents:

1. **It is a local calibration point, not a hardware constant.** If you get `0.005` seconds, that is the speed for *this exact GPU* on *this specific hardware generation* for *this specific matrix size*. Do not treat it as a static hardware limit.
2. **It serves as our relative benchmark.** Later in the course, when we implement optimizations or test different frameworks, we will use the exact same ritual (synchronize -> timer -> run -> synchronize -> timer) to measure relative speedups.
3. **Good hygiene is non-negotiable.** Without `torch.cuda.synchronize()`, timing GPU code is completely meaningless.

In the next notebook, we will use this exact ritual to compare CPU vs. GPU performance and build intuition about when the GPU actually wins. Let's move on!